In [4]:
import pandas as pd
import numpy as np
import os

# Cambia esto por la ruta de tu carpeta de UNSW-NB15
# Nota: Este dataset a veces viene en 4 archivos o un CSV grande
path_unsw = r'C:\Users\Felix\Desktop\Tesis\Datasets\Datasets unificados\UNSW-NB15-V3.csv'

try:
    # 1. Carga del dataset
    df_unsw = pd.read_csv(path_unsw, low_memory=False)
    print(f"✅ UNSW-NB15 cargado. Filas: {len(df_unsw):,}, Columnas: {len(df_unsw.columns)}")

    # 2. Limpieza de nombres y eliminación de columnas innecesarias
    df_unsw.columns = df_unsw.columns.str.strip().str.lower()
    
    # El 'id' suele introducir sesgo (el modelo aprende el orden de llegada, no el ataque)
    if 'id' in df_unsw.columns:
        df_unsw = df_unsw.drop(columns=['id'])
        print("🗑️ Columna 'id' eliminada.")

    # 3. Manejo de Infinitos y Nulos
    cols_num = df_unsw.select_dtypes(include=[np.number]).columns
    df_unsw[cols_num] = df_unsw[cols_num].replace([np.inf, -np.inf], np.nan)
    
    # En este dataset, si hay nulos, los llenamos con el máximo o la mediana
    df_unsw[cols_num] = df_unsw[cols_num].fillna(df_unsw[cols_num].max())
    
    # 4. Diagnóstico de ataques
    print("\n--- Distribución de Etiquetas (Categorías) ---")
    if 'attack_cat' in df_unsw.columns:
        print(df_unsw['ct_state_ttl'].value_counts())

    # 5. Guardar como Parquet
    output_unsw = path_unsw.replace('.csv', '.parquet')
    df_unsw.to_parquet(output_unsw, index=False)
    print(f"\n💾 UNSW-NB15 listo en: {output_unsw}")

except Exception as e:
    print(f"❌ Error al procesar UNSW-NB15: {e}")

✅ UNSW-NB15 cargado. Filas: 2,754,739, Columnas: 47

--- Distribución de Etiquetas (Categorías) ---

💾 UNSW-NB15 listo en: C:\Users\Felix\Desktop\Tesis\Datasets\Datasets unificados\UNSW-NB15-V3.parquet


In [3]:
# Corre esto solo para verificar los nombres reales
print("Columnas disponibles:")
print(df_unsw.columns.tolist())

# Busca columnas que se parezcan a 'label' o 'cat'
posibles_labels = [c for c in df_unsw.columns if 'label' in c or 'cat' in c or 'state' in c]
print(f"\nPosibles columnas de etiquetas: {posibles_labels}")

# Si encuentras la correcta (ej. 'label' o 'attack_cat'), mira sus valores:
# Reemplaza 'nombre_columna' por la que encuentres
# print(df_unsw['nombre_columna'].value_counts())

Columnas disponibles:
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload', 'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime', 'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'label']

Posibles columnas de etiquetas: ['state', 'ct_state_ttl', 'label']


In [6]:
import pyarrow.parquet as pq

# Leemos solo el esquema (los nombres de las columnas) para no gastar RAM
path_unsw = r'C:\Users\Felix\Desktop\Tesis\Datasets\Datasets unificados\UNSW-NB15-V3.parquet'
schema = pq.read_schema(path_unsw)

print("--- Columnas reales encontradas en el archivo ---")
for name in schema.names:
    print(f"'{name}'")

--- Columnas reales encontradas en el archivo ---
'srcip'
'sport'
'dstip'
'dsport'
'proto'
'state'
'dur'
'sbytes'
'dbytes'
'sttl'
'dttl'
'sloss'
'dloss'
'service'
'sload'
'dload'
'spkts'
'dpkts'
'swin'
'dwin'
'stcpb'
'dtcpb'
'smeansz'
'dmeansz'
'trans_depth'
'res_bdy_len'
'sjit'
'djit'
'stime'
'ltime'
'sintpkt'
'dintpkt'
'tcprtt'
'synack'
'ackdat'
'is_sm_ips_ports'
'ct_state_ttl'
'ct_flw_http_mthd'
'is_ftp_login'
'ct_srv_src'
'ct_srv_dst'
'ct_dst_ltm'
'ct_src_ ltm'
'ct_src_dport_ltm'
'ct_dst_sport_ltm'
'ct_dst_src_ltm'
'label'


In [10]:
import pandas as pd

path_unsw = r'C:\Users\Felix\Desktop\Tesis\Datasets\Datasets unificados\UNSW-NB15-V3.parquet'

# 1. Cargamos el archivo completo pero solo las columnas que detectemos
# Primero leemos una sola fila para mapear nombres
df_temp = pd.read_parquet(path_unsw, engine='pyarrow').head(1)
cols = df_temp.columns.tolist()

# Buscamos la columna de ataque (que contenga 'cat') y la de label
col_ataque = [c for c in cols if 'ct' in c.lower()][0]
col_label = [c for c in cols if 'label' in c.lower()][0]

print(f"🔍 Columnas detectadas: Ataque -> '{col_ataque}', Binaria -> '{col_label}'\n")

# 2. Ahora sí, cargamos solo esas dos columnas de todo el dataset
df_labels = pd.read_parquet(path_unsw, columns=[col_ataque, col_label])

# 3. Resumen de la distribución
print("--- Distribución General (Binaria) ---")
print(df_labels[col_label].value_counts())
print("-" * 30)

print("--- Distribución por Tipo de Ataque ---")
resumen = pd.DataFrame({
    'Conteo': df_labels[col_ataque].value_counts(),
    'Porcentaje (%)': (df_labels[col_ataque].value_counts(normalize=True) * 100).round(2)
})
print(resumen)

🔍 Columnas detectadas: Ataque -> 'ct_state_ttl', Binaria -> 'label'

--- Distribución General (Binaria) ---
label
benign            2218456
Generic            215481
Comb               215000
Exploits            44525
Fuzzers             24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoor             2329
Shellcode            1511
Worms                 174
Name: count, dtype: int64
------------------------------
--- Distribución por Tipo de Ataque ---
               Conteo  Porcentaje (%)
ct_state_ttl                         
0.000000      2177879           79.06
0.333333       272001            9.87
0.166667        81396            2.95
0.500000         4251            0.15
1.000000         4170            0.15
...               ...             ...
0.009828            1            0.00
0.025171            1            0.00
0.006927            1            0.00
0.074187            1            0.00
0.028562            1            0.00

[212673 